In [2]:
import glob
from tqdm.auto import tqdm
import os

In [3]:
os.makedirs('/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/whole_slice/mouse_1/zarr', exist_ok=True)
input_vsi_fn = '/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/whole_slice/mouse_1/vsi/20250815_40X_TimerMtb_BP_mice1_DAPI_TimerG_TimerR_Multichannel Z-Stack_20250815_5375.vsi'
output_zarr_fn = '/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/whole_slice/mouse_1/zarr/20250815_40X_TimerMtb_BP_mice1_DAPI_TimerG_TimerR_Multichannel Z-Stack_20250815_5375.zarr'

In [ ]:
bioformats2raw \
'/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/whole_slice/mouse_1/vsi/20250815_40X_TimerMtb_BP_mice1_DAPI_TimerG_TimerR_Multichannel Z-Stack_20250815_5375.vsi' \
'/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/whole_slice/mouse_1/zarr/20250815_40X_TimerMtb_BP_mice1_DAPI_TimerG_TimerR_Multichannel Z-Stack_20250815_5375.zarr'

In [14]:
completed_vsi = []
for input_vsi_fn in tqdm(input_vsi_fns): 
    output_zarr_fn = input_vsi_fn.replace('.vsi', '.jn.zarr')
    !/home/dayn/miniconda3/envs/godspee/bin/bioformats2raw \
    "$input_vsi_fn" "$output_zarr_fn"
    completed_vsi.append(input_vsi_fn)

  0%|          | 0/9 [00:00<?, ?it/s]

2025-10-31 15:59:15,293 [pool-1-thread-2] ERROR c.g.bioformats2raw.Converter - Failure processing chunk; resolution=0 plane=1 xx=0 yy=0 zz=0 width=1024 height=1024 depth=1
java.lang.NoClassDefFoundError: Could not initialize class org.blosc.IBloscDll
	at org.blosc.JBlosc.compressCtx(JBlosc.java:213)
	at com.bc.zarr.CompressorFactory$BloscCompressor.compress(CompressorFactory.java:346)
	at com.bc.zarr.chunk.ChunkReaderWriterImpl_Short.write(ChunkReaderWriterImpl_Short.java:83)
	at com.bc.zarr.ZarrArray.write(ZarrArray.java:239)
	at com.glencoesoftware.bioformats2raw.Converter.writeBytes(Converter.java:1790)
	at com.glencoesoftware.bioformats2raw.Converter.processChunk(Converter.java:2021)
	at com.glencoesoftware.bioformats2raw.Converter.lambda$saveResolutions$5(Converter.java:2176)
	at java.base/java.util.concurrent.ThreadPoolExecutor.runWorker(ThreadPoolExecutor.java:1128)
	at java.base/java.util.concurrent.ThreadPoolExecutor$Worker.run(ThreadPoolExecutor.java:628)
	at java.base/java.l

Exception ignored in: <function _releaseLock at 0x7dfd71796660>
Traceback (most recent call last):
  File "/home/dayn/miniconda3/envs/godspee/lib/python3.11/logging/__init__.py", line 237, in _releaseLock
KeyboardInterrupt: 

KeyboardInterrupt


KeyboardInterrupt



### Identify vsi/ets pairs for conversion

In [ ]:
# root = Path("/mnt/NEMO/home/shared/Shared - Baptiste Pradel/Histology")
root = Path("/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/whole_slice/")

# mouse dirs ≥ 7
mouse_dirs = sorted(
    [p for p in root.glob("mouse_*") if int(p.name.removeprefix("mouse_")) >= 7],
    key=lambda p: int(p.name.removeprefix("mouse_")),
)

# largest .ets per mouse
largest_ets_per_mouse = []
for mouse_dir in mouse_dirs:
    ets_files = list(mouse_dir.rglob("*.ets"))
    if ets_files:
        largest_ets_per_mouse.append(max(ets_files, key=lambda f: f.stat().st_size))

# map each .ets to its sibling .vsi using the 2nd parent folder name
def ets_to_vsi(ets_path: Path) -> Path:
    acq_dir = ets_path.parent.parent.name              # e.g. "_20250901_..._5555_"
    acq_base = acq_dir.strip("_")                      # -> "20250901_..._5555"
    mouse_dir = ets_path.parents[2]     # .../<mouse_N>/
    return mouse_dir / f"{acq_base}.vsi"               # .../<mouse_N>/<acq_base>.vsi

corresponding_vsi = []
for ets in largest_ets_per_mouse:
    vsi = ets_to_vsi(ets)
    corresponding_vsi.append(vsi if vsi.exists() else None)

# show pairs (and which are missing)
for ets, vsi in zip(largest_ets_per_mouse, corresponding_vsi):
    print(f"{ets}  ->  {vsi if vsi else 'NO MATCH'}")


## Convert to tiff

This code previous worked in two stages from terminal

~/bftools/bfconvert \
-bigtiff -compression LZW \
-series 0 \
"20250814_40X_TimerMtb_BP_mice10_DAPI_TimerG_TimerR_Multichannel Z-Stack_20250814_5375.vsi" \
"exports/mtb_slice_5375_s0.ome.tif"

bfconvert \ 
mtb_slice_5375_s0.ome.tif \
mtb_slice_5375_pyr.ome.tiff \
-bigtiff \
-pyramid-scale 2 -pyramid-resolutions 5 \
-tilex 512 -tiley 512 \
-compression LZW

I think it was the series that did it

In [ ]:
vsi_fns = [str(i) for i in corresponding_vsi]

In [ ]:
vsi_fns

#### Omitted series 0 from argument, using old version of bftools

In [ ]:
for vsi_fn in tqdm(vsi_fns):
    tif_fn = vsi_fn.replace('vsi', 'tif')
    !~/bftools/bfconvert \
    -bigtiff -compression LZW \
    -tilex 2048 -tiley 2048 \
    "$vsi_fn" \
    "$tif_fn"

## Convert to Zarr

In [ ]:
import tifffile
import dask.array as da
import zarr
from ome_zarr.io import parse_url
from ome_zarr.writer import write_image, add_metadata
from dask.diagnostics import ProgressBar
from ome_zarr.writer import write_multiscales_metadata

In [ ]:
tif_fns = glob.glob('/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/whole_slice/mouse*/tif/*')

In [ ]:
for tif_fn in tqdm(tif_fns):
    try:
        if 'ome' in tif_fn:
            print('skipping suspect corrupt')
            continue
        else:
            zarr_image = tifffile.imread(tif_fn, aszarr=True)
            dask_image = da.from_zarr(zarr_image)
            print(output_fn, dask_image.shape)
            dask_image = dask_image.transpose(1, 0, 2, 3,)  # lazy; no data copy
            # v0.5 / Zarr v3 store
            out_zarr = tif_fn.replace('.tif', '.zarr')
            store = parse_url(out_zarr, mode="w").store
            root = zarr.group(store=store)
            
            # writes data + builds a 2x YX pyramid by default
            with ProgressBar():  # optional live progress
                write_image(
                    image=dask_image,
                    group=root,
                    axes="czyx",
                    storage_options=dict(chunks=(1, 1, 512, 512)),  # (C,Z,Y,X)
                )
            
            # optional: channel labels for nicer viewing in napari/viv
            add_metadata(root, {"omero": {
                "channels": [
                    {"label": "CF405"},
                    {"label": "CF488"},
                    {"label": "CF561"},
                ]
            }})
        
            # after write_image(... axes="czyx")
            level_names = sorted(root.array_keys(), key=int)   # <-- not group_keys()
            
            axes = [
                {"name": "c", "type": "channel"},
                {"name": "z", "type": "space", "unit": "micrometer"},
                {"name": "y", "type": "space", "unit": "micrometer"},
                {"name": "x", "type": "space", "unit": "micrometer"},
            ]
            
            px_z, px_y, px_x = 2.0, 0.1625, 0.1625
            datasets = []
            for i, p in enumerate(level_names):
                datasets.append({
                    "path": p,
                    "coordinateTransformations": [
                        {"type": "scale", "scale": [1.0, px_z, px_y*(2**i), px_x*(2**i)]},  # C Z Y X
                        {"type": "translation", "translation": [0, 0, 0, 0]},
                    ]
                })
            
            write_multiscales_metadata(root, datasets=datasets, axes=axes)
    except:
        print(tif_fn, 'failed')

### Previous code

In [ ]:
for tif_fn in tqdm(tif_fns):
    zarr_image = tifffile.imread(tif_fn, aszarr=True)
    dask_image = da.from_zarr(zarr_image)
    print(output_fn, dask_image.shape)
    dask_image = dask_image.transpose(1, 0, 2, 3,)  # lazy; no data copy
    # v0.5 / Zarr v3 store
    out_zarr = output_fn.with_suffix('.zarr')

    store = parse_url(out_zarr, mode="w").store
    root = zarr.group(store=store)
    
    # writes data + builds a 2x YX pyramid by default
    with ProgressBar():  # optional live progress
        write_image(
            image=dask_image,
            group=root,
            axes="czyx",
            storage_options=dict(chunks=(1, 1, 512, 512)),  # (C,Z,Y,X)
        )
    
    # optional: channel labels for nicer viewing in napari/viv
    add_metadata(root, {"omero": {
        "channels": [
            {"label": "CF405"},
            {"label": "CF488"},
            {"label": "CF561"},
        ]
    }})

    # after write_image(... axes="czyx")
    level_names = sorted(root.array_keys(), key=int)   # <-- not group_keys()
    
    axes = [
        {"name": "c", "type": "channel"},
        {"name": "z", "type": "space", "unit": "micrometer"},
        {"name": "y", "type": "space", "unit": "micrometer"},
        {"name": "x", "type": "space", "unit": "micrometer"},
    ]
    
    px_z, px_y, px_x = 2.0, 0.1625, 0.1625
    datasets = []
    for i, p in enumerate(level_names):
        datasets.append({
            "path": p,
            "coordinateTransformations": [
                {"type": "scale", "scale": [1.0, px_z, px_y*(2**i), px_x*(2**i)]},  # C Z Y X
                {"type": "translation", "translation": [0, 0, 0, 0]},
            ]
        })
    
    write_multiscales_metadata(root, datasets=datasets, axes=axes)


# Arx

In [ ]:
import time
from pathlib import Path
from datetime import datetime, timedelta

# ---- config ----
ref_path = Path("/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/whole_slice/_20250814_40X_TimerMtb_BP_mice10_DAPI_TimerG_TimerR_Multichannel Z-Stack_20250814_5375_/stack1/frame_t_0.ets")
out_path = Path("/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/whole_slice/exports/mtb_slice_5375_s0.ome.tif")

check_every_s = 90           # polling interval
min_samples_for_eta = 5      # wait this many samples before showing ETA
window_samples = 5           # moving-average window for rate
stall_threshold_gb = 0.01    # consider “no progress” if growth < this per interval
stall_checks = 60             # consecutive stalls before we say “probably done”
target_equals_ref = True     # if False, don’t compare to ref size; just use stall logic
# -----------------

def gb(bytes_): return bytes_ / (1024**3)

ref_size_gb = gb(ref_path.stat().st_size)
print(f"Reference size: {ref_size_gb:.2f} GB (for context)\n")

sizes = []
times = []
no_progress = 0

while True:
    if out_path.exists():
        s = gb(out_path.stat().st_size)
        t = time.time()
        sizes.append(s); times.append(t)

        # progress %
        pct_str = f"{(s/ref_size_gb*100):.1f}%" if ref_size_gb > 0 else "—"

        # compute smoothed rate after we have enough points
        eta_str = "estimating…"
        rate_gb_per_min = 0.0
        if len(sizes) >= min_samples_for_eta:
            w = sizes[-window_samples:]
            wt = times[-window_samples:]
            ds = w[-1] - w[0]
            dt_min = (wt[-1] - wt[0]) / 60
            if dt_min > 0 and ds > 0:
                rate_gb_per_min = ds / dt_min

                if target_equals_ref:
                    rem = max(ref_size_gb - s, 0.0)
                    if rate_gb_per_min > 0:
                        minutes = rem / rate_gb_per_min
                        eta_time = datetime.now() + timedelta(minutes=minutes)
                        eta_str = eta_time.strftime("%H:%M:%S")
                else:
                    eta_str = "—"  # unknown final size; stall detection will stop loop

        # detect stalls (don’t trigger on first sample)
        if len(sizes) >= 2:
            delta = sizes[-1] - sizes[-2]
            if delta < stall_threshold_gb:
                no_progress += 1
            else:
                no_progress = 0

        # status line
        print(
            f"\r{datetime.now().strftime('%H:%M:%S')} | "
            f"{s:7.2f} GB ({pct_str}) | "
            f"{rate_gb_per_min:5.2f} GB/min | ETA {eta_str} | "
            f"stalls {no_progress}/{stall_checks}",
            end=""
        )

        # stop conditions:
        done_by_size = target_equals_ref and (s >= 0.995 * ref_size_gb)
        done_by_stall = no_progress >= stall_checks
        if done_by_size or done_by_stall:
            print("\n✅ Done (size target reached or growth stalled).")
            break
    else:
        print("\rWaiting for output file...", end="")

    time.sleep(check_every_s)
    